# Kaza — Part 3: Expanded LFM3 Backbone Verification

In [ ]:
!test -d /content/kaza && git -C /content/kaza pull --ff-only origin phase1-design || git clone -b phase1-design https://github.com/udohchuks/kaza.git /content/kaza
%cd /content/kaza
!pip install -q -r requirements.txt

In [ ]:
from kaza.env import print_env
from kaza.backbone import KazaBackbone, verify_step0_identity

env = print_env()
backbone = KazaBackbone(
    device=env.device,
    frozen_dtype=env.frozen_dtype,
    trainable_dtype=env.trainable_dtype,
    autocast_dtype=env.autocast_dtype,
)

trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad) / 1e6
frozen = sum(p.numel() for p in backbone.parameters() if not p.requires_grad) / 1e6
print(f"Layers: {backbone.config.num_hidden_layers} (len layer_types={len(backbone.config.layer_types)})")
print(f"New Global GQA indices: {backbone.new_layer_indices}")
print(f"Params: {trainable:.1f}M trainable (FP32) | {frozen:.1f}M frozen ({env.frozen_dtype})")
print()
print(backbone.summary())

In [ ]:
import torch

# Check #1: exact step-0 identity vs original 24-layer LFM3
max_diff = verify_step0_identity(backbone)
print(f"Check #1 (step-0 identity) max_diff = {max_diff:.6f}")

# Verify KV + ShortConv cache works across all 30 layers
x = torch.randn(1, 6, backbone.hidden_size, device=env.device, dtype=env.frozen_dtype)
with torch.no_grad():
    y_full = backbone(inputs_embeds=x, use_cache=False).last_hidden_state.float()
    pkv, outs = None, []
    for t in range(x.shape[1]):
        o = backbone(inputs_embeds=x[:, t : t + 1], past_key_values=pkv, use_cache=True)
        pkv = o.past_key_values
        outs.append(o.last_hidden_state.float())
    y_cached = torch.cat(outs, dim=1)
print(f"KV/Conv cache max_diff = {(y_full - y_cached).abs().max().item():.6f}")